# Common Crawl Collection Reporting

This notebook collates the small set of collection numbers that are useful for the Materials section of the dissertation. It reads the configured crawl years, the final processed trend output, the accepted corpus quality summaries, and the final processed corpus document file.

The goal is not to audit every operational detail. The goal is to produce stable, report-ready topline figures and to fail loudly if the synced artifacts are incomplete or internally inconsistent.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
import yaml

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs/commoncrawl_collection.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "configs/commoncrawl_collection.yaml").exists():
    raise FileNotFoundError("Run this notebook from the repository root or from notebooks/.")

CONFIG_PATH = REPO_ROOT / "configs/commoncrawl_collection.yaml"
INTERIM_COLLECTION_DIR = REPO_ROOT / "data/interim/collection"
PROCESSED_DIR = REPO_ROOT / "data/processed"
REPORT_TABLE_DIR = REPO_ROOT / "reports/tables"
PAPER_GENERATED_DIR = REPO_ROOT / "paper/generated"

## 1. Reporting Configuration

The configured crawl map is treated as the expected frame for reporting. The `PROVISIONAL` flag controls only the generated prose label, not the underlying counts.

In [ ]:
PROVISIONAL = True

with CONFIG_PATH.open() as fh:
    config = yaml.safe_load(fh)

crawl_map = pd.DataFrame(config["collection"]["crawl_map"])
configured_years = sorted(crawl_map["year"].astype(int).tolist())
year_min = min(configured_years)
year_max = max(configured_years)
year_count = len(configured_years)

crawl_map

## 2. Trend Track

Trend totals are read from the final processed trend output. That file is the appropriate source for reporting because the processed builder enforces exactly one row per configured year.

In [ ]:
trend_rates_path = PROCESSED_DIR / "trend/trend_rates.csv"
trend_rates = pd.read_csv(trend_rates_path)
trend_rates["year"] = trend_rates["year"].astype(int)

trend_years = sorted(trend_rates["year"].tolist())
trend_duplicate_years = sorted(trend_rates.loc[trend_rates["year"].duplicated(), "year"].unique().tolist())

trend_totals = {
    "wet_records_scanned": int(trend_rates["docs_scanned"].sum()),
    "wet_validated_hits": int(trend_rates["validated_hits_wet"].sum()),
    "warc_validated_hits": int(trend_rates["validated_hits_warc"].sum()),
}

trend_by_year = trend_rates.rename(
    columns={
        "docs_scanned": "wet_records_scanned",
        "validated_hits_wet": "wet_validated_hits",
        "validated_hits_warc": "warc_validated_hits",
    }
).copy()
trend_by_year["track"] = "trend"
trend_by_year["batch_count"] = 1
trend_by_year["final_documents"] = pd.NA
trend_by_year["target_documents"] = pd.NA
trend_by_year["adhd_documents"] = pd.NA
trend_by_year["autism_documents"] = pd.NA

trend_by_year[
    [
        "track",
        "year",
        "batch_count",
        "wet_records_scanned",
        "wet_validated_hits",
        "warc_validated_hits",
        "final_documents",
        "target_documents",
    ]
]

## 3. Corpus Track

Corpus totals are read from the accepted per-batch quality summaries. This keeps the notebook valid after expansion batches are added. If a year/batch has been rerun, the latest run directory for that year/batch is used.

In [ ]:
def read_metric_series(path):
    return pd.read_csv(path).set_index("metric")["value"]


def metric_number(metrics, name, default=0):
    if name not in metrics.index:
        return default
    value = pd.to_numeric(pd.Series([metrics.loc[name]]), errors="coerce").iloc[0]
    if pd.isna(value):
        return default
    return int(value) if float(value).is_integer() else float(value)


corpus_quality_dir = INTERIM_COLLECTION_DIR / "quality/corpus"
corpus_summary_paths = sorted(corpus_quality_dir.glob("*/batch_*/*/cc_collection_summary_*.csv"))

corpus_records = []
for path in corpus_summary_paths:
    year_part, batch_part, runid_part, _ = path.relative_to(corpus_quality_dir).parts
    metrics = read_metric_series(path)
    corpus_records.append(
        {
            "year": int(year_part),
            "batch": batch_part,
            "batch_number": int(batch_part.replace("batch_", "")),
            "runid": runid_part,
            "summary_path": str(path.relative_to(REPO_ROOT)),
            "wet_records_scanned": metric_number(metrics, "docs_scanned"),
            "wet_validated_hits": metric_number(metrics, "validated_hits_wet"),
            "warc_validated_hits": metric_number(metrics, "validated_hits_warc"),
            "final_documents": metric_number(metrics, "final.doc_count"),
            "target_documents": metric_number(metrics, "final.term_role.target.doc_count"),
            "adhd_documents": metric_number(metrics, "final.term_group.adhd.doc_count"),
            "autism_documents": metric_number(metrics, "final.term_group.autism.doc_count"),
        }
    )

corpus_batch_summaries = pd.DataFrame(corpus_records)
if corpus_batch_summaries.empty:
    raise FileNotFoundError(f"No corpus quality summaries found under {corpus_quality_dir}")

latest_corpus_batches = (
    corpus_batch_summaries.sort_values(["year", "batch_number", "runid"])
    .drop_duplicates(["year", "batch_number"], keep="last")
    .reset_index(drop=True)
)

corpus_by_year = (
    latest_corpus_batches.groupby("year", as_index=False)
    .agg(
        batch_count=("batch_number", "nunique"),
        wet_records_scanned=("wet_records_scanned", "sum"),
        wet_validated_hits=("wet_validated_hits", "sum"),
        warc_validated_hits=("warc_validated_hits", "sum"),
        final_documents=("final_documents", "sum"),
        target_documents=("target_documents", "sum"),
        adhd_documents=("adhd_documents", "sum"),
        autism_documents=("autism_documents", "sum"),
    )
)
corpus_by_year["track"] = "corpus"

corpus_totals = {
    "wet_records_scanned": int(corpus_by_year["wet_records_scanned"].sum()),
    "wet_validated_hits": int(corpus_by_year["wet_validated_hits"].sum()),
    "warc_validated_hits": int(corpus_by_year["warc_validated_hits"].sum()),
    "final_documents": int(corpus_by_year["final_documents"].sum()),
    "target_documents": int(corpus_by_year["target_documents"].sum()),
    "adhd_documents": int(corpus_by_year["adhd_documents"].sum()),
    "autism_documents": int(corpus_by_year["autism_documents"].sum()),
}

corpus_by_year[
    [
        "track",
        "year",
        "batch_count",
        "wet_records_scanned",
        "wet_validated_hits",
        "warc_validated_hits",
        "final_documents",
        "target_documents",
        "adhd_documents",
        "autism_documents",
    ]
]

## 4. Processed Corpus Checks

The final corpus parquet is checked against the accepted batch summaries. This catches stale processed outputs after reruns or expansion batches.

In [ ]:
processed_corpus_path = PROCESSED_DIR / "corpus/corpus_documents.parquet"
processed_corpus = pd.read_parquet(
    processed_corpus_path,
    columns=["crawl_id", "url", "term_roles", "source_corpus_path"],
)

processed_duplicate_url_count = int(processed_corpus.duplicated(["crawl_id", "url"]).sum())
role_text = processed_corpus["term_roles"].fillna("").astype(str)
processed_target_document_count = int(role_text.str.contains(r"(?:^|\|)target(?:$|\|)", regex=True).sum())
processed_baseline_document_count = int(role_text.str.contains(r"(?:^|\|)baseline(?:$|\|)", regex=True).sum())
processed_source_count = int(processed_corpus["source_corpus_path"].nunique())

processed_summary = pd.DataFrame(
    [
        {
            "processed_rows": len(processed_corpus),
            "duplicate_crawl_url_rows": processed_duplicate_url_count,
            "target_documents": processed_target_document_count,
            "baseline_documents": processed_baseline_document_count,
            "source_corpus_paths": processed_source_count,
        }
    ]
)

processed_summary

## 5. Target-Term Detail

The main Materials section should usually use only the target-group totals. This table is still useful as a compact appendix or internal check because it shows which specific terms drive target coverage.

In [ ]:
corpus_term_paths = sorted(corpus_quality_dir.glob("*/batch_*/*/cc_collection_term_summary_*.csv"))
term_frames = []
for path in corpus_term_paths:
    year_part, batch_part, runid_part, _ = path.relative_to(corpus_quality_dir).parts
    frame = pd.read_csv(path)
    frame["year"] = int(year_part)
    frame["batch"] = batch_part
    frame["batch_number"] = int(batch_part.replace("batch_", ""))
    frame["runid"] = runid_part
    term_frames.append(frame)

corpus_term_summaries = pd.concat(term_frames, ignore_index=True)
latest_term_keys = latest_corpus_batches[["year", "batch_number", "runid"]].drop_duplicates()
latest_corpus_terms = corpus_term_summaries.merge(
    latest_term_keys,
    on=["year", "batch_number", "runid"],
    how="inner",
)

target_terms_by_year = (
    latest_corpus_terms.loc[latest_corpus_terms["term_role"] == "target"]
    .pivot_table(
        index="year",
        columns="matched_term",
        values="dedup_representative_hits",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

target_terms_by_year

## 6. Consistency Checks

These checks protect the paper numbers from partial syncs, stale processed outputs, and accidental duplicate years.

In [ ]:
checks = pd.DataFrame(
    [
        {
            "check": "trend years match configured crawl map",
            "passed": trend_years == configured_years,
            "detail": f"found={trend_years}; expected={configured_years}",
        },
        {
            "check": "trend has no duplicate years",
            "passed": not trend_duplicate_years,
            "detail": f"duplicate_years={trend_duplicate_years}",
        },
        {
            "check": "corpus summaries cover every configured year",
            "passed": sorted(corpus_by_year["year"].tolist()) == configured_years,
            "detail": f"found={sorted(corpus_by_year['year'].tolist())}; expected={configured_years}",
        },
        {
            "check": "processed corpus has no duplicate crawl_id/url rows",
            "passed": processed_duplicate_url_count == 0,
            "detail": f"duplicate_rows={processed_duplicate_url_count}",
        },
        {
            "check": "processed corpus row count matches accepted quality summaries",
            "passed": len(processed_corpus) == corpus_totals["final_documents"],
            "detail": f"processed={len(processed_corpus)}; summaries={corpus_totals['final_documents']}",
        },
        {
            "check": "processed target document count matches accepted quality summaries",
            "passed": processed_target_document_count == corpus_totals["target_documents"],
            "detail": f"processed={processed_target_document_count}; summaries={corpus_totals['target_documents']}",
        },
    ]
)

if not checks["passed"].all():
    display(checks)
    failed = checks.loc[~checks["passed"], "check"].tolist()
    raise AssertionError(f"Collection reporting checks failed: {failed}")

checks

## 7. Report Outputs

The CSV files provide auditable tables. The generated TeX file provides reusable macros for the paper draft, and the plain-text snippet provides a copyable sentence for the Materials section.

In [ ]:
def fmt_int(value):
    return f"{int(value):,}"


def fmt_million(value):
    return f"{value / 1_000_000:.1f} million"


status_suffix = " (provisional)" if PROVISIONAL else ""

collection_by_year = pd.concat(
    [
        trend_by_year[
            [
                "track",
                "year",
                "batch_count",
                "wet_records_scanned",
                "wet_validated_hits",
                "warc_validated_hits",
                "final_documents",
                "target_documents",
                "adhd_documents",
                "autism_documents",
            ]
        ],
        corpus_by_year[
            [
                "track",
                "year",
                "batch_count",
                "wet_records_scanned",
                "wet_validated_hits",
                "warc_validated_hits",
                "final_documents",
                "target_documents",
                "adhd_documents",
                "autism_documents",
            ]
        ],
    ],
    ignore_index=True,
)
integer_columns = [
    "batch_count",
    "wet_records_scanned",
    "wet_validated_hits",
    "warc_validated_hits",
    "final_documents",
    "target_documents",
    "adhd_documents",
    "autism_documents",
]
for column in integer_columns:
    collection_by_year[column] = pd.to_numeric(collection_by_year[column], errors="coerce").astype("Int64")

collection_topline = pd.DataFrame(
    [
        {
            "track": "trend",
            "year_count": year_count,
            "year_range": f"{year_min}-{year_max}",
            "wet_records_scanned": trend_totals["wet_records_scanned"],
            "wet_validated_hits": trend_totals["wet_validated_hits"],
            "warc_validated_hits": trend_totals["warc_validated_hits"],
            "final_documents": pd.NA,
            "target_documents": pd.NA,
            "adhd_documents": pd.NA,
            "autism_documents": pd.NA,
            "report_note": "Fixed-effort annual trend track; final documents are not the trend denominator.",
        },
        {
            "track": "corpus",
            "year_count": year_count,
            "year_range": f"{year_min}-{year_max}",
            "wet_records_scanned": corpus_totals["wet_records_scanned"],
            "wet_validated_hits": corpus_totals["wet_validated_hits"],
            "warc_validated_hits": corpus_totals["warc_validated_hits"],
            "final_documents": corpus_totals["final_documents"],
            "target_documents": corpus_totals["target_documents"],
            "adhd_documents": corpus_totals["adhd_documents"],
            "autism_documents": corpus_totals["autism_documents"],
            "report_note": "Quality-gated corpus track for downstream semantic and contextual analysis.",
        },
    ]
)

materials_sentence = (
    f"Across {year_count} annual Common Crawl snapshots from {year_min} to {year_max}, "
    f"the trend track scans {fmt_million(trend_totals['wet_records_scanned'])} WET records "
    f"and retains {fmt_int(trend_totals['warc_validated_hits'])} WARC-validated term hits; "
    f"the corpus track scans {fmt_million(corpus_totals['wet_records_scanned'])} WET records "
    f"and yields {fmt_int(corpus_totals['final_documents'])} analysis-ready documents, "
    f"including {fmt_int(corpus_totals['target_documents'])} target documents "
    f"({fmt_int(corpus_totals['adhd_documents'])} ADHD; "
    f"{fmt_int(corpus_totals['autism_documents'])} autism){status_suffix}."
)

tex_lines = [
    "% Auto-generated by notebooks/03_commoncrawl_collection_reporting.ipynb.",
    "% Re-run the notebook after collection artifacts change.",
    f"\\newcommand{{\\ccCollectionStatus}}{{{status_suffix.strip()}}}",
    f"\\newcommand{{\\ccYearCount}}{{{year_count}}}",
    f"\\newcommand{{\\ccYearRange}}{{{year_min}--{year_max}}}",
    f"\\newcommand{{\\ccTrendWetRecordsScanned}}{{{fmt_million(trend_totals['wet_records_scanned'])}}}",
    f"\\newcommand{{\\ccTrendWarcValidatedHits}}{{{fmt_int(trend_totals['warc_validated_hits'])}}}",
    f"\\newcommand{{\\ccCorpusWetRecordsScanned}}{{{fmt_million(corpus_totals['wet_records_scanned'])}}}",
    f"\\newcommand{{\\ccCorpusFinalDocuments}}{{{fmt_int(corpus_totals['final_documents'])}}}",
    f"\\newcommand{{\\ccCorpusTargetDocuments}}{{{fmt_int(corpus_totals['target_documents'])}}}",
    f"\\newcommand{{\\ccCorpusAdhdDocuments}}{{{fmt_int(corpus_totals['adhd_documents'])}}}",
    f"\\newcommand{{\\ccCorpusAutismDocuments}}{{{fmt_int(corpus_totals['autism_documents'])}}}",
]

REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
PAPER_GENERATED_DIR.mkdir(parents=True, exist_ok=True)

collection_topline.to_csv(REPORT_TABLE_DIR / "commoncrawl_collection_topline.csv", index=False)
collection_by_year.to_csv(REPORT_TABLE_DIR / "commoncrawl_collection_by_year.csv", index=False)
target_terms_by_year.to_csv(REPORT_TABLE_DIR / "commoncrawl_corpus_target_terms_by_year.csv", index=False)
checks.to_csv(REPORT_TABLE_DIR / "commoncrawl_collection_reporting_checks.csv", index=False)
(REPORT_TABLE_DIR / "commoncrawl_collection_materials_sentence.txt").write_text(materials_sentence + "\n")
(REPORT_TABLE_DIR / "commoncrawl_collection_topline.json").write_text(
    json.dumps(collection_topline.to_dict(orient="records"), indent=2) + "\n"
)
(PAPER_GENERATED_DIR / "commoncrawl_collection_numbers.tex").write_text("\n".join(tex_lines) + "\n")

print(materials_sentence)
collection_topline